# Experiment 6: Building a Knowledge Base with the Qdrant Vector Database

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To create a production-grade knowledge base by ingesting structured and unstructured data into a vector database (Qdrant; Pinecone is an equivalent alternative), to implement metadata-filtered semantic search, and to benchmark query latency and recall at different index configurations.

## 2. Objectives

- To understand what a vector database is and why it is used for a knowledge base.
- To ingest unstructured text (embedded as vectors) together with structured metadata (the payload).
- To perform semantic search and metadata-filtered semantic search.
- To create payload indexes and observe their effect on filtered queries.
- To benchmark latency and recall for different HNSW index settings.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- Docker (to run the Qdrant server) or a free Qdrant Cloud cluster
- Required Python libraries: `qdrant-client`, `sentence-transformers`, `pandas`, `numpy`

Start the Qdrant server (one command, from a terminal):

`docker run -d --name lab-qdrant -p 6333:6333 qdrant/qdrant`

## 4. Theory

### A. Vector database

A vector database stores embeddings and finds the nearest ones to a query vector quickly. Each stored item is a **point** with an id, a **vector** (the embedding of the unstructured text) and a **payload** (structured metadata such as year, category or source). Points live in a **collection**.

### B. Structured and unstructured data together

A knowledge base needs both. The vector captures the meaning of the text ("papers about training without labels"), and the payload lets us restrict the search ("only cs.LG, from 2015 onwards"). A filtered semantic search combines the two in one query.

### C. HNSW index and its parameters

Qdrant uses the HNSW graph index for approximate nearest-neighbour search.

| Parameter | Where | Effect |
|---|---|---|
| `m` | index build | Number of links per node. Larger m gives better recall and more memory. |
| `ef_construct` | index build | Search width while building the graph. Larger gives a better graph and a slower build. |
| `hnsw_ef` | query | Search width at query time. Larger gives higher recall and higher latency. |

Recall@10 is the fraction of the true 10 nearest neighbours (found by an exact search) that the approximate search also returns.

### D. Pinecone

Pinecone offers the same ideas as a managed cloud service: an index with a dimension and metric, `upsert` of vectors with metadata, and `query` with a metadata filter. Only the client calls differ.

## 5. Algorithm / Procedure

1. Start the Qdrant server and install the libraries.
2. Load the structured and unstructured data (arXiv paper titles, abstracts and metadata).
3. Embed the text with SentenceTransformers.
4. Create a collection and upload the points with their payloads.
5. Run a semantic search and filtered searches.
6. Create payload indexes and compare filtered query time.
7. Build collections with different HNSW settings and measure latency and recall.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q qdrant-client sentence-transformers pandas numpy

### Step 2: Load the data and create embeddings

Each record has unstructured text (title and abstract) and structured fields (arXiv id, year, categories and others). The text is embedded with `all-MiniLM-L6-v2`.

In [2]:
import json, time, warnings, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 180)

papers = [json.loads(line) for line in open("data/arxiv_sample.jsonl")]
print(len(papers), "papers | fields:", list(papers[0]))

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model.max_seq_length = 128
vectors = model.encode([p["title"] + ". " + p["abstract"] for p in papers], batch_size=64,
                       normalize_embeddings=True, show_progress_bar=False)
print("Embeddings:", vectors.shape)

11762 papers | fields: ['arxiv_id', 'title', 'abstract', 'year', 'month', 'categories', 'primary_category', 'n_authors', 'has_journal_ref', 'n_versions']


Embeddings: (11762, 384)


### Step 3: Create a collection and upload the points

`VectorParams` fixes the vector size and distance metric. The payload is the structured metadata. The last 100 papers are kept out of the collection and used later as unseen test queries. Three optimizer settings make Qdrant build one HNSW graph for this small collection and use it for every search: `indexing_threshold=1000` (build the graph from 1,000 KB of vectors instead of the default 20,000 KB), `default_segment_number=1` (a single segment) and a small `full_scan_threshold` (never fall back to a brute-force scan).

In [3]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")

N = len(papers) - 100            # the last 100 papers are held out as unseen queries

def make_collection(name, m=16, ef_construct=100):
    if client.collection_exists(name):
        client.delete_collection(name)
    client.create_collection(
        name, vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(m=m, ef_construct=ef_construct, full_scan_threshold=10),
        optimizers_config=models.OptimizersConfigDiff(indexing_threshold=1000, default_segment_number=1))
    for i in range(0, N, 500):
        client.upsert(name, wait=True, points=[
            models.PointStruct(id=j, vector=vectors[j].tolist(),
                               payload={k: papers[j][k] for k in ["arxiv_id", "title", "year", "categories", "primary_category", "has_journal_ref"]})
            for j in range(i, min(i + 500, N))])
    while client.get_collection(name).status != models.CollectionStatus.GREEN:   # wait until indexing is finished
        time.sleep(1)

make_collection("arxiv")
print("Points stored:", client.count("arxiv").count)

Points stored: 11662


### Step 4: Semantic search and metadata-filtered semantic search

`search()` embeds the query text and asks Qdrant for the nearest points; an optional `query_filter` restricts the candidates by payload.

In [4]:
F = models.FieldCondition

def search(text, k=5, flt=None, collection="arxiv"):
    hits = client.query_points(collection, query=model.encode(text, normalize_embeddings=True).tolist(),
                               query_filter=flt, limit=k, with_payload=True).points
    return pd.DataFrame([{"score": round(h.score, 3), "year": h.payload["year"], "category": h.payload["primary_category"],
                          "title": h.payload["title"][:60]} for h in hits])

search("training neural networks without labelled data")

,score,year,category,title
0,0.551,2021,cs.CV,A Closer Look at Self-training for Zero-Label Semantic Segme
1,0.538,2019,cs.LG,Self-Supervised Generalisation with Meta Auxiliary Learning
2,0.530,2021,cs.CV,SimPLE: Similar Pseudo Label Exploitation for Semi-Supervise
3,0.491,2018,cs.CV,Classification-Reconstruction Learning for Open-Set Recognit
4,0.479,2021,cs.LG,Understanding Dynamics of Nonlinear Representation Learning


In [5]:
only_lg_recent = models.Filter(must=[F(key="primary_category", match=models.MatchValue(value="cs.LG")),
                                     F(key="year", range=models.Range(gte=2015))])
search("training neural networks without labelled data", flt=only_lg_recent)

,score,year,category,title
0,0.538,2019,cs.LG,Self-Supervised Generalisation with Meta Auxiliary Learning
1,0.479,2021,cs.LG,Understanding Dynamics of Nonlinear Representation Learning
2,0.477,2020,cs.LG,Probabilistic Decoupling of Labels in Classification
3,0.461,2021,cs.LG,"Zorro: Valid, Sparse, and Stable Explanations in Graph Neura"
4,0.461,2018,cs.LG,Generative Low-Shot Network Expansion


### Step 5: Payload indexes

Filtering without a payload index forces Qdrant to check the payload of many points. A payload index on a field makes the filter cheap. We time a selective filter (category `cs.CL`, year 2018 or later) before and after creating the indexes.

In [6]:
selective = models.Filter(must=[F(key="primary_category", match=models.MatchValue(value="cs.CL")),
                                F(key="year", range=models.Range(gte=2018))])
qvecs = vectors[N:]            # the 100 held-out papers

def ms_per_query(flt, reps=3):
    best = 1e9
    for _ in range(reps):
        t0 = time.time()
        for v in qvecs:
            client.query_points("arxiv", query=v.tolist(), query_filter=flt, limit=10)
        best = min(best, (time.time() - t0) / len(qvecs) * 1000)
    return round(best, 2)

no_index = ms_per_query(selective)
client.create_payload_index("arxiv", "primary_category", models.PayloadSchemaType.KEYWORD)
client.create_payload_index("arxiv", "year", models.PayloadSchemaType.INTEGER)
with_index = ms_per_query(selective)
print(f"Filtered query: {no_index} ms without payload indexes, {with_index} ms with them")

Filtered query: 47.13 ms without payload indexes, 6.07 ms with them


### Step 6: Benchmark HNSW configurations

For each combination of build parameters (`m`, `ef_construct`) a collection is built. Every collection is then queried with the 100 held-out vectors at several `hnsw_ef` values. Recall@10 compares the result with an exact search (`exact=True`), and latency is the mean time per query.

In [7]:
def benchmark(name, ef):
    got, t0 = 0, time.time()
    approx = [client.query_points(name, query=v.tolist(), limit=10, search_params=models.SearchParams(hnsw_ef=ef)).points for v in qvecs]
    ms = (time.time() - t0) / len(qvecs) * 1000
    for v, a in zip(qvecs, approx):
        exact = client.query_points(name, query=v.tolist(), limit=10, search_params=models.SearchParams(exact=True)).points
        got += len({p.id for p in a} & {p.id for p in exact}) / 10
    return round(got / len(qvecs), 3), round(ms, 2)

rows = []
for m, efc in [(2, 4), (4, 16), (16, 100)]:
    name = f"bench_m{m}"
    make_collection(name, m=m, ef_construct=efc)
    for ef in [10, 32, 64, 256]:
        recall, ms = benchmark(name, ef)
        rows.append({"m": m, "ef_construct": efc, "hnsw_ef": ef, "Recall@10": recall, "ms per query": ms})
bench = pd.DataFrame(rows)
bench

,m,ef_construct,hnsw_ef,Recall@10,ms per query
0,2,4,10,0.063,5.63
1,2,4,32,0.090,5.16
2,2,4,64,0.115,5.36
3,2,4,256,0.213,5.88
4,4,16,10,0.695,6.39
5,4,16,32,0.854,5.35
6,4,16,64,0.931,5.18
7,4,16,256,0.971,5.44
8,16,100,10,0.897,6.27
9,16,100,32,0.986,5.36


### Step 7: Clean up

In [8]:
for name in ["bench_m2", "bench_m4", "bench_m16"]:
    client.delete_collection(name)
print(client.get_collections().collections)

[CollectionDescription(name='arxiv')]


**Using Pinecone instead.** The same knowledge base can be built on Pinecone with a free API key. The calls correspond one to one:

```python
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=PINECONE_API_KEY)
pc.create_index("arxiv", dimension=384, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index("arxiv")
index.upsert([(p["arxiv_id"], vectors[i].tolist(), {"year": p["year"], "category": p["primary_category"]}) for i, p in enumerate(papers)])
index.query(vector=query_vector, top_k=5, filter={"category": "cs.LG", "year": {"$gte": 2015}}, include_metadata=True)
```

## 7. Experimental Observations

### Observation Table 1: Semantic search with and without a metadata filter

In [9]:
q = "training neural networks without labelled data"
pd.concat({"Without filter": search(q, 4), "category = cs.LG and year >= 2015": search(q, 4, only_lg_recent)}, axis=1)

Without filter                                                                              category = cs.LG and year >= 2015                 \
           score  year category                                                         title                             score  year category   
0          0.551  2021    cs.CV  A Closer Look at Self-training for Zero-Label Semantic Segme                             0.538  2019    cs.LG   
1          0.538  2019    cs.LG   Self-Supervised Generalisation with Meta Auxiliary Learning                             0.479  2021    cs.LG   
2          0.530  2021    cs.CV  SimPLE: Similar Pseudo Label Exploitation for Semi-Supervise                             0.477  2020    cs.LG   
3          0.491  2018    cs.CV  Classification-Reconstruction Learning for Open-Set Recognit                             0.461  2021    cs.LG   

                                                                 
                                                          title  
0   Self-Supervised Generalisation with Meta Auxiliary Learning  
1  Understanding Dynamics of Nonlinear Representation Learning   
2          Probabilistic Decoupling of Labels in Classification  
3  Zorro: Valid, Sparse, and Stable Explanations in Graph Neura

### Observation Table 2: Filtered query time with and without payload indexes

In [10]:
pd.DataFrame({"ms per query": [no_index, with_index]}, index=["No payload index", "Payload indexes on category and year"])

,ms per query
No payload index,47.13
Payload indexes on category and year,6.07


### Observation Table 3: Recall and latency for HNSW settings

In [11]:
bench.pivot(index="hnsw_ef", columns=["m", "ef_construct"], values=["Recall@10", "ms per query"])

Recall@10               ms per query            
m                   2      4      16           2     4     16
ef_construct       4      16     100          4     16    100
hnsw_ef                                                      
10               0.063  0.695  0.897         5.63  6.39  6.27
32               0.090  0.854  0.986         5.16  5.35  5.36
64               0.115  0.931  0.996         5.36  5.18  5.51
256              0.213  0.971  1.000         5.88  5.44  5.97

**Observations**

- 11,662 arXiv papers were stored in Qdrant as points: a 384-number vector made from the title and abstract, and a payload with the structured fields (year, category, arXiv id).
- The query "training neural networks without labelled data" returned papers from computer vision and machine learning. Adding the filter `category = cs.LG and year >= 2015` removed the computer-vision results, and every returned paper then satisfied the filter, with the closest ones still ranked by meaning.
- Payload indexes made the selective filter (cs.CL and year 2018 or later) about eight times faster: 47.1 ms per query without them and 6.1 ms with them.
- Index settings control recall. With the weakest graph (m = 2, ef_construct = 4) Recall@10 stayed between 0.06 and 0.21 even at `hnsw_ef` = 256. With m = 4 it rose from 0.70 to 0.97, and with m = 16 and ef_construct = 100 it was 0.90 at `hnsw_ef` = 10, 0.99 at 32 and 1.00 at 256.
- Raising `hnsw_ef` improves recall but costs time. Here the per-query time stayed at about 5 to 6 ms for all settings because the collection is small and the time is dominated by the network call to the server. On collections with millions of vectors the gap between the settings grows much larger.
- A good configuration for this data is m = 16 with `hnsw_ef` of about 32 to 64, which gives a recall above 0.98 at the same latency as the cheaper settings.

## 8. Result

A knowledge base of 11,662 arXiv papers was created in the Qdrant vector database, with the text embedded as vectors and the structured fields stored as payload. Semantic search and metadata-filtered semantic search were performed, and payload indexes reduced the time of a selective filtered query from 47.1 ms to 6.1 ms. Benchmarking three HNSW configurations showed Recall@10 from 0.06 to 1.00: the setting m = 16, ef_construct = 100 reached 0.986 at hnsw_ef = 32 and 1.000 at hnsw_ef = 256, with a query time of about 5 to 6 ms in all cases.

## 9. Conclusion

A vector database turns a pile of documents into a searchable knowledge base: the embedding of the unstructured text supports search by meaning, and the structured payload supports exact constraints in the same query. Payload indexes keep filtered queries fast, and the HNSW parameters give a controllable trade-off between recall and latency. The best configuration is chosen by measuring recall against an exact search on the actual data, as done here.